In [1]:
import cv2
import json
import ollama
import pandas as pd

from pathlib import Path
from paddleocr import PaddleOCR
from concurrent.futures import ThreadPoolExecutor
from concurrent.futures import as_completed

In [2]:
ocr = PaddleOCR(
    use_angle_cls=True,
    lang="en"
)

[2026/06/12 16:12:25] ppocr DEBUG: Namespace(help='==SUPPRESS==', use_gpu=False, use_xpu=False, use_npu=False, use_mlu=False, ir_optim=True, use_tensorrt=False, min_subgraph_size=15, precision='fp32', gpu_mem=500, gpu_id=0, image_dir=None, page_num=0, det_algorithm='DB', det_model_dir='C:\\Users\\anshu/.paddleocr/whl\\det\\en\\en_PP-OCRv3_det_infer', det_limit_side_len=960, det_limit_type='max', det_box_type='quad', det_db_thresh=0.3, det_db_box_thresh=0.6, det_db_unclip_ratio=1.5, max_batch_size=10, use_dilation=False, det_db_score_mode='fast', det_east_score_thresh=0.8, det_east_cover_thresh=0.1, det_east_nms_thresh=0.2, det_sast_score_thresh=0.5, det_sast_nms_thresh=0.2, det_pse_thresh=0, det_pse_box_thresh=0.85, det_pse_min_area=16, det_pse_scale=1, scales=[8, 16, 32], alpha=1.0, beta=1.0, fourier_degree=5, rec_algorithm='SVTR_LCNet', rec_model_dir='C:\\Users\\anshu/.paddleocr/whl\\rec\\en\\en_PP-OCRv4_rec_infer', rec_image_inverse=True, rec_image_shape='3, 48, 320', rec_batch_num=

In [3]:
def run_ocr(img):

    result = ocr.ocr(img)

    if not result:
        return "", 0

    if result[0] is None:
        return "", 0

    words = []
    confs = []

    for line in result[0]:

        words.append(line[1][0])
        confs.append(line[1][1])

    text = " ".join(words)

    confidence = (
        sum(confs)/len(confs)
        if len(confs)
        else 0
    )

    return text, confidence

In [4]:
def get_rotations(image):

    return {

        "original": image,

        "90_cw":
        cv2.rotate(
            image,
            cv2.ROTATE_90_CLOCKWISE
        ),

        "90_ccw":
        cv2.rotate(
            image,
            cv2.ROTATE_90_COUNTERCLOCKWISE
        ),

        "180":
        cv2.rotate(
            image,
            cv2.ROTATE_180
        )
    }

In [5]:
def get_best_ocr_text(image_path):

    image = cv2.imread(image_path)

    if image is None:
        return None

    rotations = get_rotations(image)

    best = None
    best_score = -1

    for name, rotated in rotations.items():

        text, conf = run_ocr(rotated)

        score = len(text) * conf

        if score > best_score:

            best_score = score

            best = {

                "text":
                text.upper(),

                "confidence":
                conf,

                "rotation":
                name
            }

    return best

In [6]:
def identify_book_with_gemma(
    image_path,
    ocr_text
):

    prompt = f"""
You are identifying a book from a bookshelf.

OCR text:

{ocr_text}

The OCR may contain severe mistakes.

Use BOTH:
1. The image
2. The OCR text

Most books belong to fantasy or science fiction authors.

Examples include:
Brandon Sanderson
Terry Pratchett
Anthony Ryan
Peter F. Hamilton
John Gwynne
Guy Gavriel Kay
Jay Kristoff
Andrzej Sapkowski
Ann Leckie
Alastair Reynolds

First reconstruct the likely spine text.

Then identify the exact book.

Return ONLY JSON:

{{
"title":"",
"author":"",
"corrected_spine_text":"",
"reasoning":"",
"confidence":0
}}
"""

    response = ollama.chat(

        model="gemma4:31b-cloud",

        messages=[
            {
                "role":"user",
                "content":prompt,
                "images":[image_path]
            }
        ]
    )

    return response["message"]["content"]

In [7]:
def parse_response(text):

    try:

        start = text.find("{")
        end = text.rfind("}") + 1

        return json.loads(
            text[start:end]
        )

    except:

        return {

            "title":"UNKNOWN",
            "author":"UNKNOWN",
            "confidence":0
        }

In [8]:
def process_spine(spine_path):

    try:

        best = get_best_ocr_text(
            spine_path
        )

        if best is None:

            return {

                "file": spine_path,
                "title": "UNKNOWN"
            }

        response = identify_book_with_gemma(

            spine_path,

            best["text"]
        )

        book = parse_response(
            response
        )

        return {

            "file":
            spine_path,

            "ocr":
            best["text"],

            "title":
            book.get(
                "title",
                ""
            ),

            "author":
            book.get(
                "author",
                ""
            ),

            "corrected_spine_text":
            book.get(
                "corrected_spine_text",
                ""
            ),

            "confidence":
            book.get(
                "confidence",
                0
            )
        }

    except Exception as e:

        return {

            "file":
            spine_path,

            "title":
            "ERROR",

            "error":
            str(e)
        }

In [9]:
spines = sorted(

    Path(
        "../outputs/book_spines"
    ).glob("*.jpg")

)

print(
    len(spines)
)

165


In [10]:
results = []

with ThreadPoolExecutor(
    max_workers=5
) as executor:

    futures = {

        executor.submit(
            process_spine,
            str(spine)
        ): spine

        for spine in spines
    }

    completed = 0

    for future in as_completed(
        futures
    ):

        completed += 1

        result = future.result()

        results.append(
            result
        )

        print(
            completed,
            "/",
            len(spines),
            result.get(
                "title",
                ""
            )
        )

[2026/06/12 16:12:26] ppocr DEBUG: dt_boxes num : 0, elapsed : 0.1800248622894287
[2026/06/12 16:12:26] ppocr DEBUG: dt_boxes num : 0, elapsed : 0.25654149055480957
[2026/06/12 16:12:26] ppocr DEBUG: cls num  : 0, elapsed : 0
[2026/06/12 16:12:26] ppocr DEBUG: cls num  : 0, elapsed : 0
[2026/06/12 16:12:26] ppocr DEBUG: rec_res num  : 0, elapsed : 0.0
[2026/06/12 16:12:26] ppocr DEBUG: dt_boxes num : 1, elapsed : 0.13602423667907715
[2026/06/12 16:12:26] ppocr DEBUG: rec_res num  : 0, elapsed : 0.0
[2026/06/12 16:12:26] ppocr DEBUG: cls num  : 1, elapsed : 0.1315164566040039
[2026/06/12 16:12:26] ppocr DEBUG: rec_res num  : 1, elapsed : 0.12393522262573242
[2026/06/12 16:12:26] ppocr DEBUG: dt_boxes num : 2, elapsed : 0.3379702568054199
[2026/06/12 16:12:26] ppocr DEBUG: dt_boxes num : 1, elapsed : 0.10129165649414062
[2026/06/12 16:12:26] ppocr DEBUG: cls num  : 2, elapsed : 0.04451441764831543
[2026/06/12 16:12:26] ppocr DEBUG: dt_boxes num : 3, elapsed : 0.4032573699951172
[2026/06/

In [11]:
df = pd.DataFrame(
    results
)

df.head()

,file,title,error,ocr,author,corrected_spine_text,confidence
0,..\outputs\book_spines\spine_101.jpg,ERROR,(PreconditionNotMet) Tensor holds no memory. C...,NaN,NaN,NaN,NaN
1,..\outputs\book_spines\spine_102.jpg,ERROR,index 1 is out of bounds for axis 0 with size 1,NaN,NaN,NaN,NaN
2,..\outputs\book_spines\spine_0.jpg,ERROR,(PreconditionNotMet) Tensor holds no memory. C...,NaN,NaN,NaN,NaN
3,..\outputs\book_spines\spine_103.jpg,ERROR,index 4 is out of bounds for axis 0 with size 4,NaN,NaN,NaN,NaN
4,..\outputs\book_spines\spine_100.jpg,Wintersmith,NaN,KEINSRZLNLA,Terry Pratchett,WINTERSMITH Terry Pratchett,1.0


In [12]:
df.to_csv(

    "../outputs/detected_books.csv",

    index=False
)

print(
    "Saved",
    len(df),
    "books"
)

Saved 165 books


In [13]:
df[
    [
        "title",
        "author",
        "confidence"
    ]
].sample(
    20
)

,title,author,confidence
69,The Grace of Kings,Ken Liu,0.90
124,The Truth,Terry Pratchett,0.90
31,The Eye of the World,Robert Jordan,0.95
72,The Last King,Robert F. Gilbert,0.70
94,Tigana,Guy Gavriel Kay,1.00
13,Tailored Realities,Brandon Sanderson,1.00
91,Mother of Learning,Domagoj Kurmaic,1.00
138,The Strength of the Few,James Islington,1.00
99,The Fifth Elephant,Terry Pratchett,1.00
164,Moving Pictures,Terry Pratchett,1.00
